# Subclustering Immune cells

In [ ]:
library(Signac)
library(Seurat)
library(rtracklayer)
library(dplyr)
library(Matrix)
library(ggplot2)
library(cowplot)
require(biomaRt)
library(DoubletFinder)
library(RColorBrewer)
library(stringr)
set.seed(1234)

In [ ]:
my_c <- c('#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4', '#f5674a', '#C79FEF', '#DAA520', '#A0522D')

# Data loading

In [ ]:
annotation <- import('genes2.gtf')

In [ ]:
annotation$gene_biotype[1:1780455] = annotation$gene_type[1:1780455]
seqlevelsStyle(annotation) <- "UCSC" #Support for other than Ensembl seqlevel style

In [ ]:
annotation$tx_id = annotation$transcript_id

In [ ]:
#load fragments data
P11_sample1_frags <- 'P11_sample1_atac_fragments.tsv.gz'

P11_sample2_frags <- 'P11_sample2_atac_fragments.tsv.gz'

P11_sample3_frags <- 'P11_sample3_atac_fragments.tsv.gz'

P60_sample1_frags <- 'P60_sample1_atac_fragments.tsv.gz'

P60_sample2_frags <- 'P60_sample2_atac_fragments.tsv.gz'

P60_TTR_sample3_frags <- 'P60_TTR_sample3_atac_fragments.tsv.gz'

In [ ]:
sample_names <- c('P11_sample1', 'P11_sample2', 'P11_sample3', 'P60_sample1', 'P60_sample2', 'P60_TTR_sample3')

In [ ]:
frags_path <- c(P11_sample1_frags, P11_sample2_frags, P11_sample3_frags, P60_sample1_frags, P60_sample2_frags, 
           P60_TTR_sample3_frags)

In [ ]:
raw <- readRDS('Merged_multiome.rds')

In [ ]:
filtered <- readRDS('Integrated_multiome_clusters_joint_linked_cell_type.rds')

In [ ]:
raw

In [ ]:
filtered

# Subset data into only Immune cells

In [ ]:
immune_filt <- subset(filtered, subset = predicted.id == 'Immune cells')

In [ ]:
immune_filt

In [ ]:
immune <- subset(raw, cells = rownames(immune_filt@meta.data))

In [ ]:
immune

In [ ]:
frags <- Fragments(immune)

In [ ]:
Fragments(immune) <- NULL

In [ ]:
for (i in 1:length(frags)){
    frags[[i]] <- UpdatePath(frags[[i]], new.path = frags_path[[i]])
}

In [ ]:
Fragments(immune) <- frags

In [ ]:
saveRDS(immune, 'Immune_multiome.rds')

In [ ]:
immune <- readRDS('Immune_multiome.rds')

In [ ]:
immune@assays

In [ ]:
unique(immune$Sample)

In [ ]:
unique(immune$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
immune$nucleosome_group <- ifelse(immune$nucleosome_signal > 2.0, 'NS > 2.0', 'NS < 2.0')

frag_hist <- FragmentHistogram(object = immune, group.by = 'nucleosome_group', region = 'chr1-1-10000000')

frag_hist

ggsave(file="Immune_frag_hist.svg", plot=frag_hist, width=8, height=4)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(immune, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Immune_all_vln_prefilter.svg", plot=vln, width=15, height=8)


In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(immune, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("", sample, "_vln_prefilter.svg")

    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

# Filtering

In [ ]:
table(immune$orig.ident)

In [ ]:
DefaultAssay(immune) <- "RNA"

Since we already have the peaks assay we can delete the ATAC assay, which allows to save compuattional resources 

In [ ]:
samples.list <- SplitObject(immune, split.by = 'Sample')

In [ ]:
samples.list

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
immune <- immune %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(immune, ndims = 50)

pca

ggsave(file="Immune_bff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
immune <- RunUMAP(immune, dims = 1:14, reduction = 'pca', n.neighbors = 15)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(immune, reduction = 'umap', group.by = 'Sample', pt.size = 1.5, raster = TRUE, cols = my_c)

dpl

ggsave(file="Immune_bff_RNA_UMAP.svg", plot=dpl, width=8, height=4)

**NOTE**:The functions *get_density* and *DensityScatter* are from the packages Signac V 1.10, since we are using the version 1.4 we copied only the necessary function to plot the wanted plots.

In [ ]:
get_density <- function(x, y, ...) {
  if (!requireNamespace("MASS", quietly = TRUE)) {
    stop("Please install MASS: install.packages('MASS')")
  }
  if (!is.numeric(x = x) | !is.numeric(x = y)) {
      stop("Must supply numeric values")
  }
  dens <- MASS::kde2d(x = x, y = y, ...)
  ix <- findInterval(x = x, vec = dens$x)
  iy <- findInterval(x = y, vec = dens$y)
  ii <- cbind(ix, iy)
  return(dens$z[ii])
}

In [ ]:
DensityScatter <- function(
    object,
    x,
    y,
    log_x = FALSE,
    log_y = FALSE,
    quantiles = NULL
) {
    md <- object[[]]
    if (!(x %in% colnames(x = md))) {
        stop(x, " not found")
    }
    if (!(y %in% colnames(x = md))) {
        stop(y, " not found")
    }
    log10p <- function(x) {
        return(log10(x = x + 1))
    }
    null_fn <- function(x) {
        return(x)
    }
    logfnx <- ifelse(test = log_x, yes = log10p, no = null_fn)
    logfny <- ifelse(test = log_y, yes = log10p, no = null_fn)
    # logfny <- null_fn
    md$Density <- get_density(
        x = logfnx(md[[x]]),
        y = logfny(md[[y]]),
        n = 1000,
        h = c(1,1)
    )
    md <- md[order(md$Density), ]
    # quantiles
    use_quantile <- FALSE
    if (!is.null(x = quantiles)) {
        use_quantile <- TRUE
        # set default if TRUE passed
        if (is.logical(x = quantiles)) {
            if (quantiles) {
                quantiles <- c(5, 10, 90, 95)
            } else {
                use_quantile <- FALSE
            }
        }
        # make sure integers between 0 and 100
        if (!(all(quantiles >= 0) & all(quantiles <= 100))) {
            warning("Quantile values must be between 0 and 100",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
        if (any(sapply(X = quantiles, FUN = function(x) x %% 1 != 0))) {
            warning("Quantile values must be integers",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
    }
    if (use_quantile) {
        # convert to string
        quantiles <- paste0(quantiles, '%')

        # quantiles for x and y
        x_quant <- quantile(x = md[[x]], probs = seq(0, 1, 0.01))
        y_quant <- quantile(x = md[[y]], probs = seq(0, 1, 0.01))
        xlines <- x_quant[quantiles]
        ylines <- y_quant[quantiles]

        # round
        xlines <- round(x = xlines, digits = 2)
        ylines <- round(x = ylines, digits = 2)
    }        
    p <- ggplot(
           data = md,
           mapping = aes(x = .data[[x]], y = .data[[y]], color = .data[["Density"]])
         ) +
           geom_point(size = 1) +
           scale_color_viridis_c(option = "B") +
           theme_bw()
    if (log_x) {
        p <- p + scale_x_log10()
    }
    if (log_y) {
        p <- p + scale_y_log10()
    }
    if (use_quantile) {
        p <- p +
          geom_vline(xintercept = unname(obj = xlines), color = "red") +
          geom_hline(yintercept = unname(obj = ylines), color = "red") +
          labs(title = "Quantiles",
               subtitle = paste0(x, ": ", paste0(names(x = xlines), ":",
                                                 xlines, collapse = " "), "\n",
                                 y, ": ", paste0(names(x = ylines), ":",
                                                 ylines, collapse = " ")))
    }
    return(p)
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(immune, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_RNA', y = 'nFeature_RNA', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl_RNA.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(immune, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_ATAC', y = 'TSS.enrichment', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
nCount_RNA_max <- c(3700, 3000, 6500, 3400, 2500, 1300)

nCount_RNA_min <- c(250, 250, 250, 250, 250, 250)

nFeature_RNA_max <- c(1900, 1750, 4500, 1750, 2300, 800)

nFeature_RNA_min <- c(250, 250, 250, 250, 250, 250)

nCount_ATAC_max <- c(6500, 10000, 15000, 12500, 12500, 2350)

nCount_ATAC_min <- c(400, 400, 400, 450, 600, 200)

nFeature_ATAC_max <- c(3000, 4500, 5000, 4500, 6000, 1100)

nFeature_ATAC_min <- c(250, 250, 250, 250, 250, 250)

nucleosome_signal_max <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

TSS.enrichment_min <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0)

pct_mito_max <- c(0.3, 0.3, 0.3, 1.3, 0.6, 1.2)

pct_hb_max <- c(0.075, 0.075, 0.06, 0.05, 0.06, 0.05)


In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
cells_to_keep <- c()

In [ ]:
for (i in 1:length(sample_names)){
    sample <- sample_names[i]
    print(sample)

    s <- subset(immune, subset = Sample == sample)

    print(s)
    
    print(nCount_RNA_max[i])
    print(nCount_ATAC_max[i])
    print(nCount_RNA_min[i])
    print(nCount_ATAC_min[i])
    print(nFeature_RNA_max[i])
    print(nFeature_ATAC_max[i])
    print(nFeature_RNA_min[i])
    print(nFeature_ATAC_min[i])
    print(nucleosome_signal_max[i])
    print(TSS.enrichment_min[i])
    print(pct_mito_max[i])
    print(pct_hb_max[i])
      
      
    s <-  subset(s, subset=nCount_RNA < nCount_RNA_max[i] &
                      nCount_ATAC < as.numeric(nCount_ATAC_max[i]) &
                      nCount_RNA > as.numeric(nCount_RNA_min[i]) &
                      nCount_ATAC > as.numeric(nCount_ATAC_min[i]) &
                      nFeature_RNA < as.numeric(nFeature_RNA_max[i]) &
                      nFeature_ATAC < as.numeric(nFeature_ATAC_max[i]) &
                      nFeature_RNA > as.numeric(nFeature_RNA_min[i]) &
                      nFeature_ATAC > as.numeric(nFeature_ATAC_min[i]) &
                      nucleosome_signal < as.numeric(nucleosome_signal_max[i]) &
                      TSS.enrichment > as.numeric(TSS.enrichment_min[i]) &
                      pct_mito < as.numeric(pct_mito_max[i]) &
                      pct_hb < as.numeric(pct_hb_max[i]))
    
    cells_to_keep <- c(cells_to_keep, colnames(s))
    
    print(length(colnames(s)))
    print(length(cells_to_keep))
    
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    print(vln)

    plot_name = paste("", sample, "_vln_afterfilter.svg")

    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
}

In [ ]:
length(cells_to_keep)

In [ ]:
immune

In [ ]:
immune_filt <- subset(immune, cells = cells_to_keep)

In [ ]:
length(unique(colnames(immune_filt)))

In [ ]:
table(immune_filt$orig.ident)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(immune_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Immune_all_vln_after_filter.svg", plot=vln, width=15, height=8)


In [ ]:
saveRDS(immune_filt, 'Immune_filt_multiome.rds')

# Call peaks

In [ ]:
immune_filt <- readRDS('Immune_filt_multiome.rds')

In [ ]:
immune_filt

In [ ]:
DefaultAssay(immune_filt) <- 'ATAC'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=immune_filt,
                outdir='callpeaks', cleanup=FALSE)

In [ ]:
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(peaks, ranges = blacklist_mm10, invert = TRUE)

In [ ]:
macs2_counts <- FeatureMatrix(fragments = Fragments(immune_filt), features = peaks, cells = colnames(immune_filt))

In [ ]:
immune_filt[['peaks']] <- CreateChromatinAssay(counts = macs2_counts, sep = c('-', '-'), fragments = Fragments(immune_filt), annotation = annotation)

In [ ]:
immune_filt@assays

In [ ]:
DefaultAssay(immune_filt) <- 'peaks'

In [ ]:
immune_filt[['ATAC']] <- NULL

In [ ]:
saveRDS(immune_filt, 'Immune_immune_filt_multiome_peaks.rds')

# FRiP

In [ ]:
immune_filt <- FRiP(immune_filt, assay = 'peaks', total.fragments = "nCount_ATAC", col.name = 'FRiP', verbose = TRUE)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 300)

vln <- VlnPlot(immune_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'FRiP', 'pct_mito', 'pct_hb'), 
        ncol=3, pt.size=0, col = my_c) 

vln

ggsave(file="Immune_immune_filt_vln_pre_filter_frip.svg", plot=vln, width=15, height=12)


In [ ]:
saveRDS(immune_filt, 'Immune_immune_filt_multiome_peaks.rds')

# Downstream analysis

In [ ]:
immune_filt <- readRDS('Immune_immune_filt_multiome_peaks.rds')

In [ ]:
DefaultAssay(immune_filt) <- "RNA"

## RNA

In [ ]:
immune_filt@assays

In [ ]:
table(immune_filt$orig.ident)

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
immune_filt <- immune_filt %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData(vars.to.regress = vars_to_regress) %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(immune_filt, ndims = 50)

pca

ggsave(file="Immune_aff_pca.svg", plot=pca, width=6, height=4)

In [ ]:
immune_filt <- RunUMAP(immune_filt, dims = 1:14, reduction = 'pca', n.neighbors = 15, reduction.name="umap_RNA")

In [ ]:
unique(immune_filt$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
before_umap_s <- DimPlot(immune_filt, reduction = 'umap_RNA', group.by = 'Sample', pt.size = 3.5, raster = TRUE, cols = my_c)

before_umap_s

ggsave(file="Immune_aff_Sample_UMAP.svg", plot=before_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(immune_filt, reduction = 'umap_RNA', group.by = 'Age', pt.size = 3.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Immune_aff_Age_UMAP.svg", plot=before_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(immune_filt, reduction = 'umap_RNA', group.by = 'Batch', pt.size = 3.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Immune_aff_Batch_UMAP.svg", plot=before_umap_a, width=6, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(immune_filt, group.by.vars = 'Sample', reduction.save = 'harmonyRNA')

In [ ]:
integrated

In [ ]:
integrated <- RunUMAP(integrated, reduction = 'harmonyRNA', dims = 1:14, n.neighbors = 15, reduction.name="harmony_RNA")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
after_umap_s <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Sample', pt.size = 3.5, raster = TRUE, cols = my_c)

after_umap_s

ggsave(file="Harmony_RNA_sample_UMAP.svg", plot=after_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_a <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Age', pt.size = 3.5, raster = TRUE, cols = my_c[2:3])

after_umap_a

ggsave(file="Harmony_RNA_Age_UMAP.svg", plot=after_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_b <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Batch', pt.size = 3.5, raster = TRUE, cols = my_c[2:3])

after_umap_b

ggsave(file="Harmony_RNA_Batch_UMAP.svg", plot=after_umap_b, width=6, height=4)


In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_RNA", shuffle=TRUE, cols=my_c, ncol=3, pt.size=3.5, label=FALSE)

dpl

ggsave(file="Harmony_each_sample_UMAP.svg", plot=dpl, width=24, height=12)

### Clustering

In [ ]:
integrated

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- integrated %>% FindNeighbors(reduction = 'harmonyRNA', dims = 1:14) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=3.5, group.by = 'RNA_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Immune_integrated_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
integrated <- SetIdent(integrated, value = 'RNA_snn_res.0.1')

In [ ]:
integrated_predicted.markers <- FindAllMarkers(integrated, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

In [ ]:
saveRDS(integrated, 'Immune_integrated_multiome_clusters.rds')

# ATAC

In [ ]:
integrated <- readRDS('Immune_integrated_multiome_clusters.rds')

In [ ]:
DefaultAssay(integrated) <- "peaks"

In [ ]:
integrated <- FindTopFeatures(integrated, assay = "peaks", min.cutoff = 10)

In [ ]:
integrated <- RunTFIDF(integrated, assay="peaks")

In [ ]:
integrated <- RunSVD(integrated, assay="peaks")

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
dpc <- DepthCor(integrated, n=50)

dpc

ggsave(file="Merged_depthCorr.svg", plot=dpc, width=6, height=4)

In [ ]:
options(repr.plot.width=6, repr.plot.height=4, repr.plot.res = 200)
ebl <- ElbowPlot(integrated, reduction = "lsi", n=50)

ebl

ggsave(file="Merged_depthCorr_ebl.svg", plot=ebl, width=6, height=4)

In [ ]:
integrated <- FindNeighbors(integrated, reduction = 'lsi', dims = 2:20, graph.name="snn_peaks")

In [ ]:
integrated <- RunUMAP(integrated, dims = 2:20, reduction = 'lsi', reduction.name="umap_Peaks")

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Sample", reduction="umap_Peaks", pt.size = 3.5, raster = TRUE, cols = my_c, label=FALSE)

dpl 

ggsave(file="Merged_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="umap_Peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=3.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Merged_Peaks_each_sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Batch", reduction="umap_Peaks", pt.size = 3.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_Peaks_Batch_UMAP.svg", plot=dpl, width=7, height=4)

In [ ]:
options(repr.plot.width=7, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Age", reduction="umap_Peaks", pt.size = 3.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Merged_Peaks_Age_UMAP.svg", plot=dpl, width=7, height=4)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(integrated, group.by.vars = 'Sample', reduction = 'lsi', assay.use = 'peaks', reduction.save = 'harmonyPeaks', project.dim = FALSE)

In [ ]:
# re-compute the UMAP using corrected LSI embeddings
integrated <- RunUMAP(integrated, dims = 2:20, reduction = 'harmonyPeaks', reduction.name = 'harmony_Peaks')

In [ ]:
options(repr.plot.width=8, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by = 'Sample', pt.size = 3.5, raster=TRUE, cols=my_c, reduction = 'harmony_Peaks')

dpl

ggsave(file="Immune_integrated_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=5)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=3.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="Immune_integrated_Peaks_each_Sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by="Batch", reduction="harmony_Peaks", pt.size = 3.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Immune_integrated_Peaks_Batch_UMAP.svg", plot=dpl, width=7, height=5)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)
dpl <- DimPlot(integrated, group.by="Age", reduction="harmony_Peaks", pt.size = 3.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="Immune_integrated_Peaks_Age_UMAP.svg", plot=dpl, width=7, height=5)

### Clustering

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
integrated

In [ ]:
integrated <- integrated %>% FindNeighbors(reduction = 'harmonyPeaks', dims = 2:20) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_Peaks", pt.size=3.5, group.by = 'peaks_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Immune_integrated_Peaks_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
saveRDS(integrated, 'Immune_integrated_multiome_clusters.rds')

# Cell type annotation

In [ ]:
integrated <- readRDS('Immune_integrated_multiome_clusters.rds')

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- SetIdent(integrated, value = 'RNA_snn_res.0.1')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

In [ ]:
marker_genes <- c('Folr1', 'Clic6', 'Krt18', 'Ttr', 'Dcn', 'Itga8', 'Col3a1', 'Flt1', 'Plvap', 'Pecam1', 'Cd86', 'Tank', 
                  'Cd83', 'Dscam', 'Ptprz1', 'Gria2')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 14, repr.plot.res = 300)
vln <- VlnPlot(object = integrated, features = marker_genes, pt.size=0.5, cols = my_c, ncol = 3)

vln

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 20, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_RNA', features = marker_genes, min.cutoff = 0, max.cutoff = 4, label.size = 2, 
                   ncol = 4, pt.size = 3.5, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 

In [ ]:
immune_markers <- c('Aif1', 'Adgre1', 'Fcgr1', 'Mertk',
                    'Sparc', 'Syngr1', 'Crybb1', 'Serpine2',
                    'Top2a', 'Birc5', 'Pclaf', 'Mki67',
                    'H2-DMb1', 'Cd72', 'Tnfsf9', 'Tnip3',
                    'S100a9', 'S100a8', 'Cstdc4', 'Stfa2',
                    'Emilin2', 'Ccr2', 'Ly6c2', 'Chil3',
                    'Adgre4', 'Gfpt1', 'Eno3', 'Ace',
                    'Ebf1', 'Cd79a', 'Ly6d', 'Igkc',
                    'Gzmb', 'Cd7', 'Klre1', 'Klrb1c',
                    'Cd3d', 'Lat', 'Trac', 'Cers4')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 25, repr.plot.res = 300)
vln <- VlnPlot(object = integrated, features = immune_markers, pt.size=0.8, cols = my_c, ncol = 4)

vln

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 40, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_RNA', features = immune_markers, min.cutoff = 0, max.cutoff = 4, label.size = 2, 
                   ncol = 4, pt.size = 3.5, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "RNA_snn_res.0.1", pt.size=3.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Immune_integrated_clusters_RNA.svg", plot=dpt, width=7, height=5)

In [ ]:
integrated@meta.data$RNA_snn_res.0.1 <- as.factor(integrated@meta.data$RNA_snn_res.0.1)

In [ ]:
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.1 == "0")] <- "Macrophages" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.1 == "1")] <- "Lymphocytes"

In [ ]:
integrated <- SetIdent(integrated, value = 'Cell_type')

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Cell_type", pt.size=3.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Immune_integrated_celltype_RNA.svg", plot=dpt, width=7, height=4)

In [ ]:
integrated_predicted.markers <- FindAllMarkers(integrated, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:

write.csv(integrated_predicted.markers, 'Markers_RNA_Immune.csv', row.names = TRUE)

In [ ]:
integrated_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:


options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

ggsave(file="Immune_integrated_dotplot_Cell_type.svg", plot=dtp, width=10, height=6)

In [ ]:
immune_markers <- c('Adgre1', 'Cd86', 'Abca1', 'Cd83',
                    'Skap1', 'Itk', 'Satb1', 'Vps37b')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
vln <- VlnPlot(object = integrated, features = immune_markers, pt.size=0, cols = my_c, ncol = 4, y.max=4)

vln

ggsave(file="Immune_integrated_vln_Cell_type.svg", plot=vln, width=15, height=20)

In [ ]:
integrated

In [ ]:
saveRDS(integrated, 'Immune_integrated_multiome_celltype.rds')

In [ ]:
markers <- c('Adgre1', 'Cd86', 'Abca1', 'Cd83',
                    'Skap1', 'Itk', 'Satb1', 'Vps37b')

In [ ]:
expression <- as.data.frame(t(integrated[["RNA"]]@data[markers,]))

In [ ]:
expression$Cell <- rownames(expression)
expression$Idents <- Idents(integrated)

In [ ]:
expression <- reshape2::melt(expression, id.vars = c("Cell","Idents"), measure.vars = markers,
                       variable.name = "Feat", value.name = "Expr")

In [ ]:
expression$color_group <- as.factor((as.numeric(factor(expression$Feat)) - 1) %/% 4)


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 7, repr.plot.res = 100)

# Create a grouping variable for coloring every 4 genes the same
expression$color_group <- as.factor((as.numeric(factor(expression$Feat)) - 1) %/% 4)

c <- ggplot(expression, aes(factor(Feat), Expr, fill = color_group)) +
        geom_violin(scale = "width", adjust = 1, trim = TRUE) +
        scale_y_continuous(expand = c(0, 0), position="right", labels = function(x)
                           c(rep(x = "", times = length(x)-2), x[length(x) - 1], "")) +
        facet_grid(rows = vars(Idents), scales = "free", switch = "y") +
        theme_cowplot(font_size = 12) +
        theme(legend.position = "none", panel.spacing = unit(0, "lines"),
          plot.title = element_text(hjust = 0.5),
              panel.background = element_rect(fill = NA, color = "black"),
              strip.background = element_blank(),
              strip.text = element_text(face = "bold"),
              strip.text.y.left = element_text(angle = 0),
              axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5)) + 
                           scale_fill_manual(values = c(my_c[1], my_c[11]),)

print(c)


ggsave(file="New_Immune_noDBL_all_integrated_vln_Cell_type.svg", plot=c, width=20, height=7)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Cell_type", pt.size=3.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Immune_integrated_celltype_RNA.svg", plot=dpt, width=7, height=4)

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Cell_type", pt.size=3.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_Peaks')

dpt

ggsave(file="Immune_integrated_celltype_peaks.svg", plot=dpt, width=7, height=4)

In [ ]:
saveRDS(integrated, 'Immune_integrated_multiome_celltype.rds')

# Joint viualization

In [ ]:
integrated <- readRDS('Immune_integrated_multiome_celltype.rds')

In [ ]:
integrated@reductions

In [ ]:
integrated <- FindMultiModalNeighbors(integrated, reduction.list = c('harmonyRNA', 'harmonyPeaks'),
                                       dims.list = list(1:14,2:20), knn.range = 150, verbose=TRUE)

In [ ]:
integrated <- RunUMAP(integrated, nn.name = "weighted.nn", assay = "RNA", reduction.name = 'umap_multiome',  verbose = TRUE)

In [ ]:


options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Cell_type", pt.size=3.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'umap_multiome')

dpt

ggsave(file="Immune_integrated_clusters_predictedid_Multiome.svg", plot=dpt, width=7, height=5)

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
saveRDS(integrated, 'Immune_integrated_multiome_celltype_joint.rds')

# Marker genes on different UMAP assays

In [ ]:
immune_markers <- c('Adgre1', 'Cd86', 'Abca1', 'Cd74',
                    'Skap1', 'Itk', 'Satb1', 'Vps37b')

## RNA

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_Cell_type_RNA_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_RNA', features = immune_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 3.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_geneExpression_RNA_umap.svg", plot=ftp, width=20, height=20)

## Peaks

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Cell_type', cols = my_c, reduction = 'harmony_Peaks', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_Cell_type_Peaks_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_Peaks', features = immune_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 3.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_geneExpression_Peaks_umap.svg", plot=ftp, width=20, height=20)

## Joint

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_Cell_type_Joint_umap.svg", plot=dmp, width=8, height=4)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'umap_multiome', features = immune_markers, min.cutoff = 0, 
                   max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 3.5, 
                   raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Harmony_geneExpression_Joint_umap.svg", plot=ftp, width=20, height=20)

# Linking peaks to genes

In [ ]:
integrated <- readRDS('Immune_integrated_multiome_celltype_joint.rds')

In [ ]:
library(BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
# first compute the GC content for each peak
integrated <- RegionStats(integrated, genome = BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
integrated

In [ ]:
integrated <- LinkPeaks(integrated, peak.assay = "peaks", expression.assay = "RNA")

In [ ]:
integrated

In [ ]:
saveRDS(integrated, 'Immune_integrated_multiome_celltype_joint_linked.rds')

# Finda DA peaks

In [ ]:
integrated <- readRDS('Immune_integrated_multiome_celltype_joint_linked.rds')

## Cell types

In [ ]:
DefaultAssay(integrated) <- 'peaks'
Idents(integrated) <- "Cell_type"

In [ ]:
da_peaks <- FindAllMarkers(
  object = integrated,
  test.use = 'LR',
  min.pct = 0.1
)

In [ ]:
da_peaks <- da_peaks[da_peaks$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks, 'DA_peaks_immune.csv')

## Condition

In [ ]:
Idents(integrated) <- "Age"

In [ ]:
da_peaks_age <- FindAllMarkers(
  object = integrated,
  assay = 'peaks',
  test.use = 'wilcox',
  min.pct = 0.01
)

In [ ]:
dim(da_peaks_age)

In [ ]:
da_peaks_age <- da_peaks_age[da_peaks_age$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks_age, 'DA_peaks_immune_age.csv')

In [ ]:
dim(da_peaks_age)

In [ ]:
p11_peaks <- subset(da_peaks_age, cluster == 'P11')

In [ ]:
dim(p11_peaks)

In [ ]:
p60_peaks <- subset(da_peaks_age, cluster == 'P60')

In [ ]:
dim(p60_peaks)

# Plots for figures

In [ ]:
integrated <- readRDS('Immune_integrated_multiome_celltype_joint_linked.rds')

In [ ]:
integrated$log10_ngenes <- log10(integrated$nFeature_RNA)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('log10_ngenes'), 
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,5)) + scale_fill_manual(values = c(my_c[1], my_c[11]))

vln

ggsave(file="noDBL_immune_integrated_all_vln_log10genes_Cell_type.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('pct_mito'), 
        ncol=4, pt.size=0, col = my_c) + scale_fill_manual(values = c(my_c[1], my_c[11])) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_immune_integrated_all_vln_pct_mito_Cell_type.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('log10_ngenes'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,5))

vln

ggsave(file="noDBL_immune_integrated_all_vln_log10genes_Sample.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('pct_mito'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_immune_integrated_all_vln_pct_mito_Sample.svg", plot=vln, width=14, height=5)


In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('nCount_RNA', 'log10_ngenes', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="noDBL_immune_integrated_all_vln_log10genes_pct_mito_all.svg", plot=vln, width=15, height=8)


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE, cols = c(my_c[1], my_c[11]), 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_RNA_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE, cols = c(my_c[1], my_c[11]), 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_peaks_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE, cols = c(my_c[1], my_c[11]), 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_noDBL_Cell_type_joint_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_noDBL_Age_RNA_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_noDBL_Age_peaks_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 3.5)

dmp

ggsave(file="Harmony_noDBL_Age_joint_umap.svg", plot=dmp, width=6, height=4)

## Highlight DA peaks

### Abca1

In [ ]:
cvp <- CoveragePlot(integrated, region = 'Abca1', feature = 'Abca1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = c(my_c[1], my_c[11]))
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Abca1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Itk

In [ ]:
cvp <- CoveragePlot(integrated, region = 'Itk', feature = 'Itk', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = c(my_c[1], my_c[11]))
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Itk', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

## Peaks by age

### Cd74

In [ ]:
cvp <- CoveragePlot(integrated, region = 'Cd74', feature = 'Cd74', group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 70000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_no_highlight_noDBL_", 'Cd74', "_igv_tracks_immune.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Ccl7

In [ ]:
cvp <- CoveragePlot(integrated, region = 'Ccl7', feature = 'Ccl7', group.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Age_no_highlight_noDBL_", 'Ccl7', "_igv_tracks_immune.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

# Xist counts to try to separate male and female cells 

In [ ]:
integrated <- readRDS('Immune_integrated_multiome_celltype_joint_linked.rds')

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated

In [ ]:
xist_counts <- GetAssayData(integrated, assay = "RNA", layer = "counts")[ "Xist",]

In [ ]:
integrated$'Xist-counts' <- xist_counts

In [ ]:
integrated$'Female' <- integrated$`Xist-counts` > 0

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 5)

dmp

ggsave(file="Gender_inference_Harmony_noDB_RNA_umap_immue.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 5)

dmp

ggsave(file="Gender_inference_Harmony_noDBL_peaks_umap_immue.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'umap_multiome', raster = TRUE, pt.size = 5)

dmp

ggsave(file="Gender_inference_Harmony_noDBL_joint_umap_immue.svg", plot=dmp, width=6, height=4)

# Session info

In [ ]:
print(sessioninfo::session_info())